In [1]:
import os
import boto3
from sagemaker import get_execution_role
from pprint import pprint
import json
import time
import pandas as pd

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


### Constants

In [2]:
# project
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')
# name of step function
str_name = 'christian-step-function'

Project: 20240509-christian-internship


### Hyperparameter df

In [3]:
# # make a dictionary of hyperparameters, save as df to s3, so I can re-convert it to dict in the images
# dict_hyperparameters = {
#     # data sets
#     'STR_FILENAME_TRAIN': 'df_train_noleaks_pre.gzip',
#     'STR_FILENAME_VALID': 'df_valid_noleaks_pre.gzip', # always use the full data set for the validation model
#     # ITERATIONS - define once for consistency
#     'INT_N_ITERATIONS': 1000,
#     # proportion of iterations used for early stopping
#     'PROP_EARLY_STOPPING': 0.05,
#     # tuning - 1
#     'INT_N_TUNING_JOBS_1': 100, # number of tuning jobs in the first tuning job
#     # eval metric
#     'STR_EVAL_METRIC': 'AUC',
# }

# # make df
# df = pd.DataFrame(dict_hyperparameters.items(), columns=['keys','values'])

# # save
# str_filename = 'df_hyperparameters.csv'
# str_uri = f's3://{str_project}/01_ad/02_model/01_feat_select/05_step_function/{str_filename}'
# df.to_csv(str_uri, index=False)

# # show
# df

### Write ```definition.json```

In [4]:
%%writefile definition.json

{
  "Comment": "A description of my state machine",
  "StartAt": "Tuning",
  "States": {
    "Tuning": {
      "Type": "Task",
      "Resource": "arn:aws:states:::batch:submitJob.sync",
      "Parameters": {
        "JobName": "HyperparamTuning",
        "JobDefinition": "arn:aws:batch:us-west-2:836690756591:job-definition/job-def-christian-tuning-5:1",
        "JobQueue": "arn:aws:batch:us-west-2:836690756591:job-queue/queue-christian-tuning-5",
        "ArrayProperties": {
          "Size": 10
        }
      },
      "Next": "ConcatTuning"
    },
    "ConcatTuning": {
      "Type": "Task",
      "Resource": "arn:aws:states:::lambda:invoke",
      "OutputPath": "$.Payload",
      "Parameters": {
        "FunctionName": "arn:aws:lambda:us-west-2:836690756591:function:christian-concat-tuning:$LATEST"
      },
      "Retry": [
        {
          "ErrorEquals": [
            "Lambda.ServiceException",
            "Lambda.AWSLambdaException",
            "Lambda.SdkClientException",
            "Lambda.TooManyRequestsException"
          ],
          "IntervalSeconds": 1,
          "MaxAttempts": 3,
          "BackoffRate": 2
        }
      ],
      "End": true
    }
  }
}

Overwriting definition.json


### Make string definition

In [5]:
# load it
dict_definition = json.load(open('./definition.json'))
# make into string
str_definition = json.dumps(dict_definition)

# # replace
# str_definition = str_definition.replace('"INT_N_TUNING_JOBS_1"', str(dict_hyperparameters['INT_N_TUNING_JOBS_1']))

### Create state machine

In [6]:
cls_client_sfn = boto3.client('stepfunctions')

In [7]:
# get role
str_role = get_execution_role()
print(f'Role: {str_role}')

Role: arn:aws:iam::836690756591:role/risk-ops-role


In [8]:
# list state machines
dict_response = cls_client_sfn.list_state_machines(
)
list_dict_state_machines = dict_response['stateMachines']
list_dict_state_names = [{dict_state_machine['name']: dict_state_machine['stateMachineArn']} for dict_state_machine in list_dict_state_machines]
dict_state_names = {key: val for dict_name in list_dict_state_names for key, val in dict_name.items()}
pprint(dict_state_names)

{'DemoStepFunction': 'arn:aws:states:us-west-2:836690756591:stateMachine:DemoStepFunction',
 'MyStateMachine-fldl4s6of': 'arn:aws:states:us-west-2:836690756591:stateMachine:MyStateMachine-fldl4s6of',
 'christian-step-function': 'arn:aws:states:us-west-2:836690756591:stateMachine:christian-step-function',
 'gen-xi-retro-scoring': 'arn:aws:states:us-west-2:836690756591:stateMachine:gen-xi-retro-scoring',
 'gen-xii-payload-parsing-jq': 'arn:aws:states:us-west-2:836690756591:stateMachine:gen-xii-payload-parsing-jq',
 'gen-xii-retro-scoring': 'arn:aws:states:us-west-2:836690756591:stateMachine:gen-xii-retro-scoring',
 'genxi-payload-parsing': 'arn:aws:states:us-west-2:836690756591:stateMachine:genxi-payload-parsing',
 'genxii-payload-parsing': 'arn:aws:states:us-west-2:836690756591:stateMachine:genxii-payload-parsing',
 'poc-step-genxii-lgd-lambda-boto3': 'arn:aws:states:us-west-2:836690756591:stateMachine:poc-step-genxii-lgd-lambda-boto3',
 'poc-step-genxii-pd-lambda-boto3': 'arn:aws:state

In [9]:
# get list of just names
list_str_names = [list(dict_state_names.keys())[0] for dict_state_names in list_dict_state_names]
# if our name is in there
if str_name in list_str_names:
    print(f'State machine {str_name} exists, it will be deleted')
    str_arn = dict_state_names[str_name]
    print(f'Deleting {str_arn}')
    print('')
    dict_response = cls_client_sfn.delete_state_machine(
        stateMachineArn=str_arn,
    )
    pprint(dict_response)
else:
    print(f'State machine {str_name} does not exist, so it will not be deleted')

State machine christian-step-function exists, it will be deleted
Deleting arn:aws:states:us-west-2:836690756591:stateMachine:christian-step-function

{'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-length': '2',
                                      'content-type': 'application/x-amz-json-1.0',
                                      'date': 'Wed, 12 Jun 2024 20:23:35 GMT',
                                      'x-amzn-requestid': 'e13ef567-7a45-4668-813b-584fc3c8a14d'},
                      'HTTPStatusCode': 200,
                      'RequestId': 'e13ef567-7a45-4668-813b-584fc3c8a14d',
                      'RetryAttempts': 0}}


In [10]:
# make a state machine
while True:
    try:
        dict_response = cls_client_sfn.create_state_machine(
            name=str_name,
            definition=str_definition,
            roleArn=str_role,
            type='STANDARD',
        )
        pprint(dict_response)
        break
    except:
        pass

{'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-length': '128',
                                      'content-type': 'application/x-amz-json-1.0',
                                      'date': 'Wed, 12 Jun 2024 20:29:05 GMT',
                                      'x-amzn-requestid': '05997e16-5620-4f5e-85cd-54c3acd18641'},
                      'HTTPStatusCode': 200,
                      'RequestId': '05997e16-5620-4f5e-85cd-54c3acd18641',
                      'RetryAttempts': 0},
 'creationDate': datetime.datetime(2024, 6, 12, 20, 29, 5, 925000, tzinfo=tzlocal()),
 'stateMachineArn': 'arn:aws:states:us-west-2:836690756591:stateMachine:christian-step-function'}


### Describe state machine

In [11]:
str_state_machine_arn = dict_response['stateMachineArn']
print(f'State Machine ARN: {str_state_machine_arn}')
dict_response = cls_client_sfn.describe_state_machine(
    stateMachineArn=str_state_machine_arn,
)
pprint(dict_response)

State Machine ARN: arn:aws:states:us-west-2:836690756591:stateMachine:christian-step-function
{'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-length': '1479',
                                      'content-type': 'application/x-amz-json-1.0',
                                      'date': 'Wed, 12 Jun 2024 20:29:10 GMT',
                                      'x-amzn-requestid': 'c1a3ca97-2069-4f25-84af-a2799a013015'},
                      'HTTPStatusCode': 200,
                      'RequestId': 'c1a3ca97-2069-4f25-84af-a2799a013015',
                      'RetryAttempts': 0},
 'creationDate': datetime.datetime(2024, 6, 12, 20, 29, 5, 925000, tzinfo=tzlocal()),
 'definition': '{"Comment": "A description of my state machine", "StartAt": '
               '"Tuning", "States": {"Tuning": {"Type": "Task", "Resource": '
               '"arn:aws:states:::batch:submitJob.sync", "Parameters": '
               '{"JobName": "Hyperpa

### Execute step function workflow

In [ ]:
# # start execution
# dict_response = cls_client_sfn.start_execution(
#     stateMachineArn=str_state_machine_arn,
# )

### Clean-up

In [ ]:
os.remove('./definition.json')